# Job Recommendation System - Model Evaluation
This notebook evaluates the recommendation models used in the backend. We compare three approaches:
1. **Bag of Words (BoW)**: Simple keyword matching based on word counts.
2. **TF-IDF**: Keyword matching that weights unique terms more heavily.
3. **SBERT (BGE-Base-En-v1.5)**: Deep learning based semantic embeddings for understanding meaning.

The goal is to verify the accuracy of the system as a reference for the final report.

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import math
import os

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Data Preparation
We load the data from the backend directory to ensure we are testing the exact same dataset used in production.

In [2]:
DATA_PATH = '../backend/data/data.csv'
df = pd.read_csv(DATA_PATH).fillna('')

print(f"Loaded {len(df)} jobs.")

# Pre-processing matching App.py
df['tfidf_features'] = (
    (df['job_title'] + ' ') * 3 + 
    (df['category'] + ' ') * 2 + 
    (df['skills_required'].str.replace(';', ' ') + ' ') * 3 + 
    df['job_description']
)

df['semantic_text'] = (
    "Job Title: " + df['job_title'] + ". " +
    "Category: " + df['category'] + ". " +
    "Skills: " + df['skills_required'].str.replace(';', ', ') + ". " +
    "Description: " + df['job_description']
)

## 2. Geometry-Correct Location Weighting
Using the Haversine formula and exponential decay as implemented in `App.py`.

In [3]:
COORDS_MAP = {
    "Phnom Penh": (11.5564, 104.9282),
    "Kandal": (11.4746, 104.9474),
    "Siem Reap": (13.3671, 103.8448),
    "Sihanoukville": (10.6093, 103.5296),
    "Battambang": (13.0957, 103.2022),
    "Kampong Cham": (11.9924, 105.4645),
    "Kampot": (10.5942, 104.1814),
    "Kratié": (12.4881, 106.0187),
    "Mondulkiri": (12.4558, 107.1747),
    "Preah Vihear": (13.8073, 104.9811),
    "Ratanakiri": (13.8577, 107.0125),
    "Takeo": (10.9908, 104.7846),
    "Remote": None
}

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi, dlambda = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def get_geometry_weight(user_loc, job_loc, scale=150):
    if user_loc == "All" or job_loc == "Remote" or user_loc == job_loc: return 1.0
    p1, p2 = COORDS_MAP.get(user_loc), COORDS_MAP.get(job_loc)
    if not p1 or not p2: return 0.5
    dist = haversine_distance(p1[0], p1[1], p2[0], p2[1])
    return math.exp(-dist / scale)

## 3. Train/Validation Split
We use a 95/5 split to have a substantial validation set (100+ samples).

In [4]:
df_train, df_val = train_test_split(df, test_size=0.05, random_state=42)
print(f"Training set: {len(df_train)} samples")
print(f"Validation set: {len(df_val)} samples")

## 4. Initialize Models
Mirroring the initialization in `App.py`.

In [5]:
# 1. BoW
bow_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = bow_vectorizer.fit_transform(df_train['tfidf_features'])

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_train['tfidf_features'])

# 3. SBERT (BGE Base v1.5)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
print(f"Using device: {device}")
sbert_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
sbert_embeddings = sbert_model.encode(df_train['semantic_text'].tolist(), show_progress_bar=True)

## 5. Quantitative Evaluation (Related but Not Relevant)
We evaluate each model on the validation set using a **category-based relevance** criterion.

A job is considered **"related but not relevant"** if it belongs to the **same category** as the query job but has a **different job title**. This reflects a realistic scenario: the user has a specific role in mind, but we want to measure how well the system surfaces *other roles within the same field*.

- ✅ **Related (counts as a hit):** same category, different job title
- ❌ **Not counted:** exact same job title (trivially relevant)
- ❌ **Not counted:** different category entirely

In [6]:
def evaluate():
    """
    Evaluate models using 'related but not relevant' relevance:
    A job is a hit if it belongs to the SAME CATEGORY as the query job
    but has a DIFFERENT job title (cross-title, same-field recommendation).
    """
    metrics = {
        'bow':   {'m5': 0, 'm10': 0, 'any5': 0, 'any10': 0, 'rec5': [], 'rec10': [], 'hits@1': 0, 'rr': [], 'ndcg': []},
        'tfidf': {'m5': 0, 'm10': 0, 'any5': 0, 'any10': 0, 'rec5': [], 'rec10': [], 'hits@1': 0, 'rr': [], 'ndcg': []},
        'sbert': {'m5': 0, 'm10': 0, 'any5': 0, 'any10': 0, 'rec5': [], 'rec10': [], 'hits@1': 0, 'rr': [], 'ndcg': []}
    }

    test_size    = len(df_val)
    train_cats   = df_train['category'].str.lower().str.strip().values
    train_titles = df_train['job_title'].str.lower().str.strip().values

    print(f"Evaluating {test_size} samples (related-but-not-relevant criterion)...")

    # Encode validation set once
    val_bow   = bow_vectorizer.transform(df_val['tfidf_features'])
    val_tfidf = tfidf_vectorizer.transform(df_val['tfidf_features'])
    instruction = "Represent this sentence for searching relevant passages: "
    val_sbert = sbert_model.encode(
        [instruction + q for q in df_val['semantic_text']], show_progress_bar=True
    )

    for i, (_, row) in enumerate(df_val.iterrows()):
        target_title = str(row['job_title']).lower().strip()
        target_cat   = str(row['category']).lower().strip()

        # "Related but not relevant" = same category, different title
        is_related = np.array([
            (c == target_cat and t != target_title)
            for c, t in zip(train_cats, train_titles)
        ])
        total_related = int(is_related.sum())
        if total_related == 0:
            continue  # no related jobs to evaluate against

        s_bow   = cosine_similarity(val_bow[i],                   bow_matrix).flatten()
        s_tfidf = cosine_similarity(val_tfidf[i],                tfidf_matrix).flatten()
        s_sbert = cosine_similarity(val_sbert[i].reshape(1, -1), sbert_embeddings).flatten()

        scores_dict = {'bow': s_bow, 'tfidf': s_tfidf, 'sbert': s_sbert}

        for model, scores in scores_dict.items():
            top_100_idx = scores.argsort()[::-1][:100]
            rel_top_100 = is_related[top_100_idx]

            rel_top_5  = rel_top_100[:5]
            rel_top_10 = rel_top_100[:10]

            # Hits@1
            if rel_top_100[0]:
                metrics[model]['hits@1'] += 1

            count5  = int(rel_top_5.sum())
            count10 = int(rel_top_10.sum())

            metrics[model]['m5']  += count5
            metrics[model]['m10'] += count10
            if count5  > 0: metrics[model]['any5']  += 1
            if count10 > 0: metrics[model]['any10'] += 1

            metrics[model]['rec5'].append(count5  / total_related)
            metrics[model]['rec10'].append(count10 / total_related)

            # MRR / nDCG@5: rank of first related hit
            first_hit = next(
                (rank for rank, rel in enumerate(rel_top_100, start=1) if rel), None
            )
            if first_hit is not None:
                metrics[model]['rr'].append(1.0 / first_hit)
                metrics[model]['ndcg'].append(
                    1.0 / math.log2(first_hit + 1) if first_hit <= 5 else 0.0
                )
            else:
                metrics[model]['rr'].append(0.0)
                metrics[model]['ndcg'].append(0.0)

    results = []
    for m, d in metrics.items():
        n = len(d['rec5'])  # profiles that had at least one related job
        results.append({
            'Model':              m.upper(),
            'Profiles Evaluated': n,
            'Top-1 Acc (%)':      (d['hits@1'] / n) * 100,
            'Success@5 (%)':      (d['any5']   / n) * 100,
            'Precision@5 (%)':    (d['m5']     / (n * 5))  * 100,
            'Recall@5 (%)':       np.mean(d['rec5'])        * 100,
            'Success@10 (%)':     (d['any10']  / n) * 100,
            'Precision@10 (%)':   (d['m10']    / (n * 10)) * 100,
            'Recall@10 (%)':      np.mean(d['rec10'])       * 100,
            'MRR':                np.mean(d['rr']),
            'nDCG@5':             np.mean(d['ndcg']),
        })
    return pd.DataFrame(results)

results_df = evaluate()
results_df


## 6. Result Visualization
Comparison charts for Top-5 and Top-10 performance.

In [7]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

df5 = results_df.melt(id_vars='Model', value_vars=['Success@5 (%)', 'Precision@5 (%)', 'Recall@5 (%)'])
sns.barplot(data=df5, x='Model', y='value', hue='variable', ax=ax1)
ax1.set_title('Top-5 Performance Metrics')
ax1.set_ylabel('Score (%)')
ax1.set_ylim(0, 105)
ax1.legend(title='Metric', bbox_to_anchor=(1, 1))

df10 = results_df.melt(id_vars='Model', value_vars=['Success@10 (%)', 'Precision@10 (%)', 'Recall@10 (%)'])
sns.barplot(data=df10, x='Model', y='value', hue='variable', ax=ax2)
ax2.set_title('Top-10 Performance Metrics')
ax2.set_ylabel('Score (%)')
ax2.set_ylim(0, 105)
ax2.legend(title='Metric', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

## 7. Conclusion
The evaluation demonstrates the differences between keyword-based approaches (BoW, TF-IDF) and semantic-based approaches (SBERT).

While SBERT handles semantic similarity (e.g., "Software Engineer" matches "Developer"), TF-IDF ensures that specific technical terms and proper nouns are matched correctly. This comparison helps in selecting the best single-model approach or understanding the trade-offs in search accuracy versus semantic understanding.

## 8. Extended Evaluation Metrics
In this section, we explicitly evaluate the models using **Precision@10**, **Recall@10**, **Mean Average Precision (MAP)**, and **NDCG@10** to provide a more comprehensive view of the ranking performance.

In [8]:
def calculate_map(top_titles, target_title, total_available):
    """Calculates Average Precision for a single query."""
    hits = 0
    sum_precisions = 0
    for i, title in enumerate(top_titles):
        if title == target_title:
            hits += 1
            sum_precisions += hits / (i + 1)
    # Standard MAP denominator is total relevant items
    return sum_precisions / total_available if total_available > 0 else 0

def calculate_ndcg(top_titles, target_title, total_available, k=10):
    """Calculates NDCG@k for a single query (binary relevance)."""
    dcg = 0
    for i in range(min(len(top_titles), k)):
        if top_titles[i] == target_title:
            dcg += 1 / math.log2(i + 2)
    
    # Ideal DCG: sum of 1/log2(i+2) for the number of relevant items available (capped at k)
    idcg = sum([1 / math.log2(i + 2) for i in range(min(total_available, k))])
    return dcg / idcg if idcg > 0 else 0

def evaluate_extended():
    metrics = {
        'bow': {'p10': [], 'r10': [], 'ap': [], 'ndcg10': []},
        'tfidf': {'p10': [], 'r10': [], 'ap': [], 'ndcg10': []},
        'sbert': {'p10': [], 'r10': [], 'ap': [], 'ndcg10': []}
    }
    
    test_size = len(df_val)
    title_counts = df_train['job_title'].str.lower().str.strip().value_counts().to_dict()
    
    # Consistent feature extraction
    val_bow = bow_vectorizer.transform(df_val['tfidf_features'])
    val_tfidf = tfidf_vectorizer.transform(df_val['tfidf_features'])
    instruction = "Represent this sentence for searching relevant passages: "
    val_sbert = sbert_model.encode([instruction + q for q in df_val['semantic_text']], show_progress_bar=False)
    
    for i, (_, row) in enumerate(df_val.iterrows()):
        target_title = str(row['job_title']).lower().strip()
        total_available = title_counts.get(target_title, 1)
        
        s_bow = cosine_similarity(val_bow[i], bow_matrix).flatten()
        s_tfidf = cosine_similarity(val_tfidf[i], tfidf_matrix).flatten()
        s_sbert = cosine_similarity(val_sbert[i].reshape(1, -1), sbert_embeddings).flatten()
        
        scores_dict = {'bow': s_bow, 'tfidf': s_tfidf, 'sbert': s_sbert}
        
        for model, scores in scores_dict.items():
            top_indices = scores.argsort()[::-1]
            top_titles = [str(t).lower().strip() for t in df_train.iloc[top_indices[:100]]['job_title']]
            
            # Precision@10
            hits10 = sum([1 for t in top_titles[:10] if t == target_title])
            metrics[model]['p10'].append(hits10 / 10)
            
            # Recall@10
            metrics[model]['r10'].append(hits10 / total_available)
            
            # Average Precision
            metrics[model]['ap'].append(calculate_map(top_titles, target_title, total_available))
            
            # NDCG@10
            metrics[model]['ndcg10'].append(calculate_ndcg(top_titles, target_title, total_available, k=10))
                
    results = []
    for m, d in metrics.items():
        results.append({
            'Model': m.upper(),
            'Precision@10 (%)': np.mean(d['p10']) * 100,
            'Recall@10 (%)': np.mean(d['r10']) * 100,
            'MAP (%)': np.mean(d['ap']) * 100,
            'NDCG@10 (%)': np.mean(d['ndcg10']) * 100
        })
    return pd.DataFrame(results)

extended_results_df = evaluate_extended()
extended_results_df

### 8.1. Extended Metrics Visualization
Comparison of models across ranking-specific metrics using a vibrant color palette.

In [9]:
plt.figure(figsize=(12, 7))

# Simplify metric names for the x-axis
df_plot = extended_results_df.melt(id_vars='Model', var_name='Metric', value_name='Score (%)')
df_plot['Metric'] = df_plot['Metric'].replace({
    'Precision@10 (%)': 'Precision',
    'Recall@10 (%)': 'Recall',
    'MAP (%)': 'MAP',
    'NDCG@10 (%)': 'NDCG'
})

# Define the requested color palette
palette = {'BOW': 'blue', 'TFIDF': 'green', 'SBERT': 'orange'}

# Plot with Metrics on x-axis and Model as hue (ranking models for each metric)
sns.barplot(data=df_plot, x='Metric', y='Score (%)', hue='Model', palette=palette)

plt.title('Ranking Performance Comparison k=10', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('Score (%)', fontsize=12, fontweight='bold')
plt.xlabel('Metrics', fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

## 9. User Profile Evaluation
We evaluate the system by simulating **User Profiles** (Expected Job Title + Skills).

**Relevance is now strict:** a recommended job counts as a hit **only if its title exactly matches the user's expected job title**. All other jobs — including those in the same category — are treated as **irrelevant**.

This measures whether the system can precisely surface the role the user is actually looking for.

In [10]:
def calculate_map_profile(top_titles, target_title, total_available):
    """Average Precision: relevance = exact title match only."""
    hits, p_sum = 0, 0
    for i, title in enumerate(top_titles):
        if title == target_title:
            hits += 1
            p_sum += hits / (i + 1)
    return p_sum / total_available if total_available > 0 else 0

def calculate_ndcg_profile(top_titles, target_title, total_available, k=10):
    """NDCG@k: relevance = exact title match only."""
    dcg = sum(
        1 / math.log2(i + 2)
        for i in range(min(len(top_titles), k))
        if top_titles[i] == target_title
    )
    idcg = sum(1 / math.log2(i + 2) for i in range(min(total_available, k)))
    return dcg / idcg if idcg > 0 else 0

def calculate_user_profile_metrics():
    """
    User Profile Evaluation with STRICT title-match relevance.
    A hit = recommended job title matches the user's expected job title exactly.
    All other jobs (even same category, different title) are irrelevant.
    """
    results = {'bow': [], 'tfidf': [], 'sbert': []}

    # Query: job title + skills from the user profile
    val_queries = "Job Title: " + df_val['job_title'] + ". Skills: " + df_val['skills_required']

    # How many training jobs share each title (denominator for Recall / MAP)
    title_counts = df_train['job_title'].str.lower().str.strip().value_counts().to_dict()
    train_titles = df_train['job_title'].str.lower().str.strip().values

    print(f"Evaluating {len(df_val)} User Profiles (strict title-match relevance)...")

    instruction = "Represent this sentence for searching relevant passages: "
    val_sbert_encoded = sbert_model.encode(
        [instruction + q for q in val_queries], show_progress_bar=True
    )

    for i in range(len(df_val)):
        profile      = df_val.iloc[i]
        query_text   = val_queries.iloc[i]
        target_title = str(profile['job_title']).lower().strip()

        total_available = title_counts.get(target_title, 0)
        if total_available == 0:
            continue  # target title not present in training set at all

        q_bow   = bow_vectorizer.transform([query_text])
        q_tfidf = tfidf_vectorizer.transform([query_text])
        q_sbert = val_sbert_encoded[i].reshape(1, -1)

        scores_dict = {
            'bow':   cosine_similarity(q_bow,   bow_matrix).flatten(),
            'tfidf': cosine_similarity(q_tfidf, tfidf_matrix).flatten(),
            'sbert': cosine_similarity(q_sbert, sbert_embeddings).flatten(),
        }

        for model, scores in scores_dict.items():
            sorted_indices  = scores.argsort()[::-1]
            top_100_titles  = [train_titles[idx] for idx in sorted_indices[:100]]
            top_10_titles   = top_100_titles[:10]

            # Precision@10: fraction of top-10 that match the expected title
            hits10 = sum(1 for t in top_10_titles if t == target_title)
            p10    = hits10 / 10

            # Recall@10: fraction of all available matching titles retrieved in top-10
            r10 = hits10 / total_available

            # MAP (over top-100)
            map_val = calculate_map_profile(top_100_titles, target_title, total_available)

            # NDCG@10
            ndcg10 = calculate_ndcg_profile(top_10_titles, target_title, total_available, k=10)

            results[model].append([p10, r10, map_val, ndcg10])

    return results

user_profile_results = calculate_user_profile_metrics()


In [11]:
print("Results (User Profile — Strict Title-Match Relevance):")
print(f"{'Model':<10} | {'P@10':<8} | {'R@10':<8} | {'MAP@100':<8} | {'NDCG@10':<8}")
print("-" * 65)
for model in ['bow', 'tfidf', 'sbert']:
    m = np.mean(user_profile_results[model], axis=0)
    print(f"{model.upper():<10} | {m[0]:.4f}   | {m[1]:.4f}   | {m[2]:.4f}   | {m[3]:.4f}")


### 9.1. User Profile Evaluation Visualization
Comparison of models for strict title-match recommendation accuracy.

In [12]:
profile_metrics = []
for model in ['bow', 'tfidf', 'sbert']:
    m = np.mean(user_profile_results[model], axis=0)
    profile_metrics.append({'Model': model.upper(), 'Metric': 'Precision@10', 'Score (%)': m[0] * 100})
    profile_metrics.append({'Model': model.upper(), 'Metric': 'Recall@10',    'Score (%)': m[1] * 100})
    profile_metrics.append({'Model': model.upper(), 'Metric': 'MAP@10',      'Score (%)': m[2] * 100})
    profile_metrics.append({'Model': model.upper(), 'Metric': 'NDCG@10',      'Score (%)': m[3] * 100})

plt.figure(figsize=(12, 7))
sns.barplot(
    data=pd.DataFrame(profile_metrics),
    x='Metric', y='Score (%)', hue='Model',
    palette={'BOW': 'blue', 'TFIDF': 'green', 'SBERT': 'orange'}
)
plt.title('User Profile Recommendation — Strict Title-Match Relevance', fontsize=15, fontweight='bold')
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()


## 11. User Profile Bar Chart Evaluation (Exact Title Match)

This section generates the **10 user profiles** defined in the standalone evaluation notebook and benchmarks them against all three models (BoW, TF-IDF, SBERT) using **strict exact-title-match** relevance.

A recommendation counts as a hit **only** when the recommended job title is an exact match of the user's desired title.  
Jobs in the same category with a different title are counted as **irrelevant**.

Metrics reported per model:
- **Exact listings found** – raw count of exact-title jobs in the training set  
- **Irrelevant listings** – same-category, different-title count  
- **Precision@10** – fraction of top-10 results with the exact title  
- **Recall@10** – fraction of all exact-title jobs retrieved in top-10  
- **MAP@100** – Mean Average Precision over top-100  
- **NDCG@10** – Normalised Discounted Cumulative Gain at rank 10

In [13]:
# ── 11. User Profile Bar Chart Evaluation ─────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

# 10 fixed user profiles
user_profiles = [
    {"id": 1,  "name": "Sophea Meas",    "desired": "Financial Analyst",              "category": "Banking & Finance",      "education": "Bachelor",    "experience": 2,
     "skills": "excel; risk management; sql; loan processing"},
    {"id": 2,  "name": "Dara Chann",     "desired": "Events Coordinator",             "category": "Hospitality & Tourism",  "education": "Master",      "experience": 5,
     "skills": "opera pms; teamwork; hotel management; customer service"},
    {"id": 3,  "name": "Kosal Tep",      "desired": "Ai/Ml Engineer",                 "category": "Technology & It",        "education": "Bachelor",    "experience": 3,
     "skills": "python; machine learning; sql; tensorflow; data analysis"},
    {"id": 4,  "name": "Sreymom Keo",    "desired": "Curriculum Developer",           "category": "Education & Training",   "education": "Bachelor",    "experience": 4,
     "skills": "curriculum design; lesson planning; classroom management; ms office"},
    {"id": 5,  "name": "Vibol Heng",     "desired": "Accountant",                     "category": "Accounting & Audit",     "education": "Bachelor",    "experience": 2,
     "skills": "accounting; quickbooks; financial reporting; ms office"},
    {"id": 6,  "name": "Channary Ros",   "desired": "Real Estate Analyst",            "category": "Real Estate",            "education": "High School", "experience": 1,
     "skills": "real estate law; valuation; negotiation; networking; english"},
    {"id": 7,  "name": "Bunna Sak",      "desired": "M&E Officer",                    "category": "Ngo & Development",      "education": "Bachelor",    "experience": 3,
     "skills": "community development; monitoring; evaluation; report writing"},
    {"id": 8,  "name": "Pisey Lim",      "desired": "Digital Marketing Executive",    "category": "Media & Marketing",      "education": "Bachelor",    "experience": 2,
     "skills": "social media; content creation; seo; photoshop; branding"},
    {"id": 9,  "name": "Makara Noun",    "desired": "Logistics Manager",              "category": "Logistics & Transport",  "education": "Master",      "experience": 6,
     "skills": "supply chain; logistics; warehouse management; inventory"},
    {"id": 10, "name": "Reaksmey Chan",  "desired": "Customer Service Representative","category": "Customer Service",       "education": "Bachelor",    "experience": 1,
     "skills": "customer service; crm; communication; problem solving"},
]

# ── Helper functions (same as Section 9) ──────────────────────────────────────
def _map(top_titles, target, total):
    hits, s = 0, 0.0
    for i, t in enumerate(top_titles):
        if t == target:
            hits += 1
            s += hits / (i + 1)
    return s / total if total > 0 else 0.0

def _ndcg(top_titles, target, total, k=10):
    dcg  = sum(1/math.log2(i+2) for i,t in enumerate(top_titles[:k]) if t == target)
    idcg = sum(1/math.log2(i+2) for i in range(min(total, k)))
    return dcg / idcg if idcg > 0 else 0.0

# ── Build query strings for each profile ──────────────────────────────────────
train_titles = df_train["job_title"].str.lower().str.strip().values
train_cats   = df_train["category"].str.lower().str.strip().values
title_counts = df_train["job_title"].str.lower().str.strip().value_counts().to_dict()

queries = [
    f"Job Title: {p["desired"]}. Skills: {p["skills"]}"
    for p in user_profiles
]

# ── Encode queries for all three models ───────────────────────────────────────
q_bows   = bow_vectorizer.transform(queries)
q_tfidfs = tfidf_vectorizer.transform(queries)
instruction = "Represent this sentence for searching relevant passages: "
q_sberts = sbert_model.encode([instruction + q for q in queries], show_progress_bar=True)

# ── Evaluate ──────────────────────────────────────────────────────────────────
MODEL_KEYS = ["bow", "tfidf", "sbert"]
records = []   # one row per (profile × model)

for pi, p in enumerate(user_profiles):
    target  = p["desired"].lower().strip()
    cat     = p["category"].lower().strip()
    total   = title_counts.get(target, 0)

    # Dataset counts (model-independent)
    exact_count = int((train_titles == target).sum())
    irrel_count = int(((train_cats == cat) & (train_titles != target)).sum())

    if total == 0:
        for m in MODEL_KEYS:
            records.append({"profile": p["name"], "desired": p["desired"],
                            "model": m.upper(), "exact_listings": exact_count,
                            "irrelevant_listings": irrel_count,
                            "P@10": 0, "R@10": 0, "MAP@10": 0, "NDCG@10": 0})
        continue

    scores_dict = {
        "bow":   cosine_similarity(q_bows[pi],            bow_matrix).flatten(),
        "tfidf": cosine_similarity(q_tfidfs[pi],          tfidf_matrix).flatten(),
        "sbert": cosine_similarity(q_sberts[pi].reshape(1,-1), sbert_embeddings).flatten(),
    }

    for m, scores in scores_dict.items():
        idx100 = scores.argsort()[::-1][:100]
        top100 = [train_titles[i] for i in idx100]
        top10  = top100[:10]
        hits10 = sum(1 for t in top10 if t == target)
        records.append({
            "profile":             p["name"],
            "desired":             p["desired"],
            "model":               m.upper(),
            "exact_listings":      exact_count,
            "irrelevant_listings": irrel_count,
            "P@10":                hits10 / 10,
            "R@10":                hits10 / total,
            "MAP@10":              _map(top10, target, total), 
            "NDCG@10":             _ndcg(top10, target, total),
        })

eval_df = pd.DataFrame(records)

# ── Print summary table ────────────────────────────────────────────────────────
print("=" * 80)
print("  USER PROFILE EVALUATION — EXACT TITLE MATCH (per model)")
print("=" * 80)
print(f"{"Profile":<18} {"Desired Title":<32} {"Model":<7} {"Exact":>6} {"Irrel":>6} {"P@10":>6} {"R@10":>6} {"MAP":>6} {"NDCG":>6}")
print("-" * 80)
for _, r in eval_df.iterrows():
    print(f"{r["profile"]:<18} {r["desired"]:<32} {r["model"]:<7} "
          f"{r["exact_listings"]:>6} {r["irrelevant_listings"]:>6} "
          f"{r["P@10"]:>6.3f} {r["R@10"]:>6.3f} {r["MAP@10"]:>6.3f} {r["NDCG@10"]:>6.3f}")

print()
print("Aggregate per model:")
agg = eval_df.groupby("model")[["P@10","R@10","MAP@10","NDCG@10"]].mean()
print(agg.round(4))


In [14]:
# ── Chart 1: Exact vs Irrelevant listings per profile (model-independent) ──────
profile_summary = eval_df.drop_duplicates('profile')[
    ['profile','exact_listings','irrelevant_listings']
].reset_index(drop=True)

names      = profile_summary['profile'].str.split().str[0].tolist()
exact_vals = profile_summary['exact_listings'].tolist()
irrel_vals = profile_summary['irrelevant_listings'].tolist()
x = np.arange(len(names))
w = 0.38
COLOR_E, COLOR_I = '#1D9E75', '#D85A30'

fig, axes = plt.subplots(3, 1, figsize=(14, 17), facecolor='#FAFAF8',
                          gridspec_kw={'height_ratios': [2.2, 2.2, 1.5]})

# ── Panel A: Exact vs Irrelevant ───────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#FAFAF8')
be = ax.bar(x - w/2, exact_vals, w, color=COLOR_E, label='Exact match',    zorder=3, linewidth=0)
bi = ax.bar(x + w/2, irrel_vals, w, color=COLOR_I, label='Irrelevant (same category)', zorder=3, linewidth=0)
for bar in be:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+1, str(int(h)),
            ha='center', va='bottom', fontsize=8.5,
            color=COLOR_E if h > 0 else '#999', fontweight='bold')
for bar in bi:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+1, str(int(h)),
            ha='center', va='bottom', fontsize=8.5, color=COLOR_I, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('Number of job listings', fontsize=11)
ax.set_title('A  Exact vs irrelevant listings per user profile (dataset counts)', fontsize=12, pad=10)
ax.yaxis.grid(True, color='#E0E0E0', linewidth=0.6, zorder=0); ax.set_axisbelow(True)
for sp in ax.spines.values(): sp.set_visible(False)
ax.tick_params(left=False)
ax.legend(handles=[mpatches.Patch(color=COLOR_E, label='Exact match'),
                   mpatches.Patch(color=COLOR_I, label='Irrelevant (same category)')],
          loc='upper right', frameon=False, fontsize=10)

# ── Panel B: P@10 per model per profile ───────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#FAFAF8')
MODEL_COLORS = {'BOW': '#185FA5', 'TFIDF': '#854F0B', 'SBERT': '#533AB7'}
bar_w = 0.25
offsets = {'BOW': -bar_w, 'TFIDF': 0, 'SBERT': bar_w}
profile_names_short = eval_df.drop_duplicates('profile')['profile'].str.split().str[0].tolist()
x2 = np.arange(len(profile_names_short))

for model in ['BOW', 'TFIDF', 'SBERT']:
    vals = [eval_df[(eval_df['profile'].str.split().str[0] == n) & (eval_df['model'] == model)]['P@10'].values[0]
            for n in profile_names_short]
    bars = ax2.bar(x2 + offsets[model], [v*100 for v in vals], bar_w,
                   color=MODEL_COLORS[model], label=model, zorder=3, linewidth=0)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v*100:.0f}',
                     ha='center', va='bottom', fontsize=7.5, color=MODEL_COLORS[model], fontweight='bold')

ax2.set_xticks(x2); ax2.set_xticklabels(profile_names_short, fontsize=10)
ax2.set_ylabel('Precision@10 (%)', fontsize=11)
ax2.set_title('B  Precision@10 per user profile — by model', fontsize=12, pad=10)
ax2.yaxis.grid(True, color='#E0E0E0', linewidth=0.6, zorder=0); ax2.set_axisbelow(True)
for sp in ax2.spines.values(): sp.set_visible(False)
ax2.tick_params(left=False)
ax2.legend(frameon=False, fontsize=10)

# ── Panel C: Aggregate metrics per model ──────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor('#FAFAF8')
agg_df = eval_df.groupby('model')[['P@10','R@10','MAP@100','NDCG@10']].mean().reset_index()
metric_cols = ['P@10','R@10','MAP@100','NDCG@10']
metric_labels = ['Precision@10','Recall@10','MAP@100','NDCG@10']
x3 = np.arange(len(metric_cols))
bw = 0.22
off3 = {'BOW': -bw, 'TFIDF': 0, 'SBERT': bw}
for model in ['BOW','TFIDF','SBERT']:
    row = agg_df[agg_df['model'] == model]
    if row.empty: continue
    vals = [row[c].values[0]*100 for c in metric_cols]
    bars = ax3.bar(x3 + off3[model], vals, bw,
                   color=MODEL_COLORS[model], label=model, zorder=3, linewidth=0)
    for bar, v in zip(bars, vals):
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.1f}',
                 ha='center', va='bottom', fontsize=8, color=MODEL_COLORS[model], fontweight='bold')

ax3.set_xticks(x3); ax3.set_xticklabels(metric_labels, fontsize=10)
ax3.set_ylabel('Score (%)', fontsize=11)
ax3.set_title('C  Aggregate retrieval metrics — all profiles, by model', fontsize=12, pad=10)
ax3.yaxis.grid(True, color='#E0E0E0', linewidth=0.6, zorder=0); ax3.set_axisbelow(True)
for sp in ax3.spines.values(): sp.set_visible(False)
ax3.tick_params(left=False)
ax3.legend(frameon=False, fontsize=10)

plt.tight_layout(pad=2)
plt.savefig('user_profile_barchart_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as user_profile_barchart_evaluation.png")


## 10. Manual Case Study: Qualitative Evaluation
To better understand how the models behave with real-world human queries, we test 10 manual profiles representing different sectors of the Cambodian job market.

In [15]:
manual_profiles = [
    {"name": "Delivery Driver", "query": "I am looking for a delivery driver job in Phnom Penh. I have a motorbike and know the city well.", "expected": "Logistics/Transport"},
    {"name": "Security Guard", "query": "Security guard, honest and hardworking, 1 year experience, looking for night shift.", "expected": "Security"},
    {"name": "Software Developer", "query": "Full stack developer skilled in Python, React, and SQL. 3 years of experience.", "expected": "IT/Software Development"},
    {"name": "Sales Assistant", "query": "Sales assistant for a retail shop or supermarket. Good communication and customer service.", "expected": "Sales/Retail"},
    {"name": "Waitress/Waiter", "query": "Experience in international restaurants, good English, looking for server position.", "expected": "Hotel/Hospitality"},
    {"name": "Accountant", "query": "Accountant with knowledge of QuickBooks and Excel. I can manage tax and payroll.", "expected": "Accounting/Finance"},
    {"name": "Construction Labor", "query": "Strong construction worker, experienced in site clearing and heavy lifting.", "expected": "Construction"},
    {"name": "Hotel Receptionist", "query": "Receptionist for hotel or guesthouse. Fluent in English and Chinese.", "expected": "Hotel/Hospitality"},
    {"name": "Cleaner", "query": "Office cleaner or maid. Diligent, punctual, and very tidy.", "expected": "Cleaning/Janitorial"},
    {"name": "Admin Assistant", "query": "Administrative assistant, good at typing, filing documents, and office management.", "expected": "Admin/Human Resources"}
]

def run_manual_study():
    print(f"{'PROFILE':<20} | {'MODEL':<8} | {'TOP 3 RECOMMENDATIONS (Score - Category)'}")
    print("-" * 110)
    
    instruction = "Represent this sentence for searching relevant passages: "
    hits = {'BoW': 0, 'TF-IDF': 0, 'SBERT': 0}

    for p in manual_profiles:
        query = p['query']
        expected = p['expected'].lower()
        
        # Scores
        q_bow = bow_vectorizer.transform([query])
        s_bow = cosine_similarity(q_bow, bow_matrix).flatten()
        
        q_tfidf = tfidf_vectorizer.transform([query])
        s_tfidf = cosine_similarity(q_tfidf, tfidf_matrix).flatten()
        
        q_sbert = sbert_model.encode([instruction + query], show_progress_bar=False)
        s_sbert = cosine_similarity(q_sbert.reshape(1, -1), sbert_embeddings).flatten()
        
        models = [('BoW', s_bow), ('TF-IDF', s_tfidf), ('SBERT', s_sbert)]
        
        print(f"{p['name']:<20} (Expected: {p['expected']})")
        for m_name, scores in models:
            top_idx = scores.argsort()[-3:][::-1]
            recs = []
            for i, idx in enumerate(top_idx):
                job = df_train.iloc[idx]
                if i == 0 and expected in job['category'].lower(): hits[m_name] += 1
                recs.append(f"{job['job_title']} ({scores[idx]:.2f} - {job['category']})")
            
            print(f"{'':<20} | {m_name:<8} | 1. {recs[0]}")
            print(f"{'':<20} | {'':<8} | 2. {recs[1]}")
            print(f"{'':<20} | {'':<8} | 3. {recs[2]}")
        print("-" * 110)
    
    print("SUMMARY: Category Match Accuracy")
    for m, count in hits.items():
        print(f"{m:<8}: {count}/{len(manual_profiles)} ({count/len(manual_profiles)*100:.1f}%)")

run_manual_study()

In [16]:
# ── Robust Model Performance Evaluation & Visualization ──────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def clean_columns(df):
    """Removes debugger-injected paths from column names and ensures standard unique names."""
    new_cols = []
    for col in df.columns:
        c = str(col)
        if 'MAP' in c: new_name = 'MAP @10'
        elif 'P@10' in c or 'P @10' in c: new_name = 'Precision @10'
        elif 'R@10' in c or 'R @10' in c: new_name = 'Recall @10'
        elif 'NDCG' in c: new_name = 'NDCG @10'
        else: new_name = c
        new_cols.append(new_name)
    df.columns = new_cols
    return df.loc[:, ~df.columns.duplicated()]

try:
    # 1. Access the evaluation data
    current_df = None
    if 'eval_df' in globals():
        current_df = eval_df.copy()
    elif 'records' in globals() and len(records) > 0:
        current_df = pd.DataFrame(records)
    
    if current_df is None:
        print("⚠️ Note: 'eval_df' not found in memory. Using cached results.")
        cached_data = {
            'model': ['BOW', 'SBERT', 'TFIDF'],
            'Precision @10': [0.39, 0.64, 0.47],
            'Recall @10': [0.3796, 0.6110, 0.4417],
            'MAP @10': [0.4550, 0.7218, 0.5302],
            'NDCG @10': [0.5006, 0.7231, 0.5711]
        }
        current_df = pd.DataFrame(cached_data)
        clean_df = current_df
    else:
        clean_df = clean_columns(current_df)
        
    # 2. Calculate Aggregate Metrics
    metrics = ['Precision @10', 'Recall @10', 'MAP @10', 'NDCG @10']
    existing_metrics = [m for m in metrics if m in clean_df.columns]
    aggregate = clean_df.groupby('model')[existing_metrics].mean()
    
    # Add Accuracy Label
    if 'Precision @10' in aggregate.columns:
        aggregate['Accuracy (P@10)'] = aggregate['Precision @10']
    
    # 3. Final Table Formatting
    final_cols = [c for c in ['Accuracy (P@10)'] + metrics if c in aggregate.columns]
    final_table = aggregate[final_cols] * 100
    
    print("=== Final Model Performance Comparison (All Profiles) ===")
    display(final_table.round(2).astype(str) + '%')

    # 4. PLOTTING THE BARCHART
    plt.figure(figsize=(14, 8), facecolor='#FAFAF8')
    ax = plt.gca()
    ax.set_facecolor('#FAFAF8')
    
    # Transpose for plotting: rows = metrics, columns = models
    plot_data = final_table.T
    
    x = np.arange(len(plot_data.index))
    width = 0.25
    
    # Colors requested: Red, Blue, Green
    colors = {'BOW': '#D85A30', 'TFIDF': '#185FA5', 'SBERT': '#1D9E75'}
    # Mapping labels to colors for consistency if models exist
    m_list = list(plot_data.columns)
    
    for i, model in enumerate(m_list):
        pos = x + (i - len(m_list)/2 + 0.5) * width
        vals = plot_data[model].values
        bars = ax.bar(pos, vals, width, label=model, color=colors.get(model, '#999'), zorder=3)
        
        # Add values above bins
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', 
                    fontweight='bold', fontsize=10, color=bar.get_facecolor())

    ax.set_xticks(x)
    ax.set_xticklabels(plot_data.index, fontsize=11, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title('Global Performance Comparison: BOW vs TF-IDF vs SBERT', fontsize=14, pad=20, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.legend(frameon=False, loc='upper left', fontsize=11)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6, zorder=0)
    for sp in ax.spines.values(): sp.set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    best_model = aggregate['NDCG @10'].idxmax() if 'NDCG @10' in aggregate.columns else "Unknown"
    print(f"🏆 Best Model (by Ranking Quality): {best_model}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")


## 12. Detailed Salary Distributions (Mountain Plots) by Category
This section visualizes the distribution of **Minimum**, **Average**, and **Maximum** salaries across job categories using Ridgeline plots to illustrate the "mountain range" of salary densities.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Data Preparation
salary_cols = ['salary_min', 'salary_avg', 'salary_max']
for col in salary_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter categories with at least 5 data points for better distribution curves
def get_filtered_df(df, column):
    temp_df = df.dropna(subset=[column]).copy()
    counts = temp_df['category'].value_counts()
    valid_cats = counts[counts >= 5].index
    return temp_df[temp_df['category'].isin(valid_cats)]

# 2. Reusable Mountain (Ridgeline) Plot Function
def plot_mountain_range(data, column, title_suffix):
    sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
    
    # Sort categories by median salary for the specific column
    medians = data.groupby('category')[column].median().sort_values(ascending=False)
    data['category_sorted'] = pd.Categorical(data['category'], categories=medians.index, ordered=True)
    
    # Initialize the FVerbosityacetGrid
    g = sns.FacetGrid(data, row="category_sorted", hue="category_sorted", 
                      aspect=10, height=0.8, palette="viridis")

    # Draw the densities (the mountains)
    g.map(sns.kdeplot, column, bw_adjust=.5, clip_on=False, fill=True, alpha=0.8, linewidth=1.5)
    g.map(sns.kdeplot, column, clip_on=False, color="w", lw=2, bw_adjust=.5)
    g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

    # Label categories
    def label_cats(x, color, label):
        ax = plt.gca()
        ax.text(0, .2, label, fontweight="bold", color=color,
                ha="left", va="center", transform=ax.transAxes, fontsize=10)
    
    g.map(label_cats, column)

    # Styling
    g.figure.subplots_adjust(hspace=-0.2)
    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.despine(bottom=True, left=True)
    
    plt.suptitle(f'Distribution of {title_suffix} Salary by Category', fontsize=18, fontweight='bold', y=0.98)
    g.set_xlabels(f'{title_suffix} Salary Amount', fontsize=12, fontweight='bold')
    plt.show()

# 3. Generate the 3 Plots
plot_mountain_range(get_filtered_df(df, 'salary_min'), 'salary_min', 'Minimum')
plot_mountain_range(get_filtered_df(df, 'salary_avg'), 'salary_avg', 'Average')
plot_mountain_range(get_filtered_df(df, 'salary_max'), 'salary_max', 'Maximum')

# Reset theme
sns.set_theme(style="whitegrid")

## 12. Segmented Salary Distribution Histograms (Slide Optimized)
To ensure the visualizations fit properly on presentation slides, each salary metric (Min, Avg, Max) has been split into three parts, each containing a subset of job categories.

In [19]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# 1. Data Preparation
salary_cols = ['salary_min', 'salary_avg', 'salary_max']
for col in salary_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter categories with at least 5 data points
def get_filtered_df(df, column):
    temp_df = df.dropna(subset=[column]).copy()
    counts = temp_df['category'].value_counts()
    valid_cats = counts[counts >= 5].index
    return temp_df[temp_df['category'].isin(valid_cats)]

# 2. Segmented Faceted Histogram Function
def plot_salary_histograms_split(data, column, title_suffix, base_filename):
    sns.set_theme(style="whitegrid")
    
    # Sort categories by median salary
    medians = data.groVerbosityupby('category')[column].median().sort_values(ascending=False)
    categories_ordered = medians.index.tolist()
    
    # Split into 3 parts
    n = len(categories_ordered)
    chunk_size = int(np.ceil(n / 3))
    chunks = [categories_ordered[i:i + chunk_size] for i in range(0, n, chunk_size)]
    
    for idx, chunk in enumerate(chunks):
        part_num = idx + 1
        chunk_data = data[data['category'].isin(chunk)].copy()
        chunk_data['category_sorted'] = pd.Categorical(chunk_data['category'], categories=chunk, ordered=True)
        
        # Initialize FacetGrid for this chunk
        g = sns.FacetGrid(chunk_data, row="category_sorted", aspect=4, height=1.5, sharex=True, sharey=False)
        
        # Draw Histograms
        g.map(sns.histplot, column, kde=True, color="#3498db", alpha=0.7, bins=30)
        
        # Styling
        g.set_titles(row_template="{row_name}", fontweight='bold', size=12)
        plt.subplots_adjust(top=0.92)
        g.fig.suptitle(f'{title_suffix} Salary Distribution - Part {part_num}', fontsize=16, fontweight='bold', y=0.98)
        g.set_xlabels(f'{title_suffix} Salary (USD)', fontsize=12, fontweight='bold')
        
        filename = f"{base_filename.replace('.png', '')}_part{part_num}.png"
        plt.savefig(filename, bbox_inches='tight', dpi=100)
        plt.show()

# 3. Generate Split Plots
plot_salary_histograms_split(get_filtered_df(df, 'salary_min'), 'salary_min', 'Minimum', 'salary_min.png')
plot_salary_histograms_split(get_filtered_df(df, 'salary_avg'), 'salary_avg', 'Average', 'salary_avg.png')
plot_salary_histograms_split(get_filtered_df(df, 'salary_max'), 'salary_max', 'Maximum', 'salary_max.png')

In [ ]:
# ── Robust Model Performance Evaluation & Visualization ──────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import traceback
from IPython.display import display

def clean_columns(df):
    """Renames raw metric column names to standardized display names."""
    new_cols = []
    for col in df.columns:
        c = str(col)
        if 'MAP' in c:       new_name = 'MAP @10'
        elif 'P@10' in c or 'P @10' in c: new_name = 'Precision @10'
        elif 'R@10' in c or 'R @10' in c: new_name = 'Recall @10'
        elif 'NDCG' in c:    new_name = 'NDCG @10'
        else:                new_name = c
        new_cols.append(new_name)
    df.columns = new_cols
    return df.loc[:, ~df.columns.duplicated()]

try:
    # 1. Load evaluation data
    current_df = None
    if 'eval_df' in globals() and isinstance(eval_df, pd.DataFrame):
        current_df = eval_df.copy()
    elif 'records' in globals() and isinstance(records, list) and len(records) > 0:
        current_df = pd.DataFrame(records)

    if current_df is None:
        print("⚠️  eval_df not found in memory — using cached results.")
        cached_data = {
            'model':          ['BOW',    'SBERT',  'TFIDF'],
            'Precision @10':  [0.39,     0.64,     0.47],
            'Recall @10':     [0.3796,   0.6110,   0.4417],
            'MAP @10':        [0.4550,   0.7218,   0.5302],
            'NDCG @10':       [0.5006,   0.7231,   0.5711],
        }
        current_df = pd.DataFrame(cached_data)
        clean_df = current_df
    else:
        clean_df = clean_columns(current_df)

    # 2. Aggregate per model
    metrics = ['Precision @10', 'Recall @10', 'MAP @10', 'NDCG @10']
    existing_metrics = [m for m in metrics if m in clean_df.columns]

    if not existing_metrics:
        print("❌  No metric columns found after renaming.")
        print("   Columns present:", list(clean_df.columns))
    else:
        aggregate = clean_df.groupby('model', sort=False)[existing_metrics].mean()

        # 3. Scale to % and enforce display order
        final_table = aggregate * 100
        desired_order = ['BOW', 'TFIDF', 'SBERT']
        present  = [m for m in desired_order if m in final_table.index]
        others   = [m for m in final_table.index if m not in desired_order]
        final_table = final_table.reindex(present + others)

        print("=== Final Model Performance (%) ===")
        display(final_table.round(2).astype(str) + '%')

        # 4. Bar chart
        fig, ax = plt.subplots(figsize=(14, 8), facecolor='#FAFAF8')
        ax.set_facecolor('#FAFAF8')

        plot_data = final_table.T          # rows = metrics, cols = models
        x     = np.arange(len(plot_data.index))
        width = 0.22
        colors = {'BOW': '#D85A30', 'TFIDF': '#185FA5', 'SBERT': '#1D9E75'}
        m_list = list(plot_data.columns)

        for i, model in enumerate(m_list):
            offset = (i - len(m_list) / 2 + 0.5) * width
            vals   = plot_data[model].values
            bars   = ax.bar(x + offset, vals, width,
                            label=model, color=colors.get(model, '#999'), zorder=3)
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                        f'{h:.1f}%', ha='center', va='bottom',
                        fontweight='bold', fontsize=10, color=bar.get_facecolor())

        ax.set_xticks(x)
        ax.set_xticklabels(plot_data.index, fontsize=11, fontweight='bold')
        ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
        ax.set_title('Global Performance: BOW vs TF-IDF vs SBERT',
                     fontsize=14, pad=20, fontweight='bold')
        ax.set_ylim(0, 110)
        ax.legend(frameon=False, loc='upper left', fontsize=11)
        ax.yaxis.grid(True, linestyle='--', alpha=0.6, zorder=0)
        for sp in ax.spines.values():
            sp.set_visible(False)

        plt.tight_layout()
        plt.show()

        if 'NDCG @10' in aggregate.columns:
            best = aggregate['NDCG @10'].idxmax()
            print(f"🏆 Best Model (NDCG @10): {best}")

except Exception:
    traceback.print_exc()
